# ViLexTime test2: gọi model trên Colab (lọc / viết câu hỏi)

Chọn GPU (T4 trở lên). Chạy lần lượt từng ô. Tải lên **một** file `inputs.jsonl` từ `test2_colab.py export` (mỗi dòng `item`, `system`, `user`).
Kết quả `raw.jsonl` (mỗi dòng `item`, `output`) lưu từng mục vào Drive nên ngắt giữa chừng thì chạy lại ô cuối, mục đã xong được bỏ qua.
Greedy, không lấy mẫu; model và revision ghi vào từng dòng. Bộ lọc chỉ dùng được nếu qua hiệu chuẩn ở máy (`test2_packs.py score`).

In [ ]:
%pip -q install -U 'transformers>=4.54,<5' accelerate bitsandbytes

In [ ]:
MODEL_ID = "Qwen/Qwen3-8B"      # đổi được; ghi lại trong hồ sơ kỹ thuật. 4-bit NF4.
MAX_NEW, BATCH = 400, 4
import json, os, hashlib
from pathlib import Path
from google.colab import drive, files
drive.mount("/content/drive")
OUT = Path("/content/drive/MyDrive/vilextime_test2"); OUT.mkdir(exist_ok=True)
up = files.upload(); name = next(iter(up)); Path("inputs.jsonl").write_bytes(up[name])
rows = [json.loads(l) for l in open("inputs.jsonl", encoding="utf-8") if l.strip()]
SHA = hashlib.sha256(Path("inputs.jsonl").read_bytes()).hexdigest()
RAW = OUT / f"raw_{SHA[:12]}.jsonl"
print(len(rows), "mục; sha256", SHA[:12], "->", RAW)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
tok = AutoTokenizer.from_pretrained(MODEL_ID, padding_side="left")
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, device_map="auto", quantization_config=BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16))
REV = getattr(model.config, "_commit_hash", None)

In [ ]:
done = {json.loads(l)["item"] for l in open(RAW, encoding="utf-8")} if RAW.exists() else set()
todo = [r for r in rows if r["item"] not in done]
print("còn", len(todo))
def prompt(r):
    m = [{"role": "system", "content": r["system"]}, {"role": "user", "content": r["user"]}]
    return tok.apply_chat_template(m, tokenize=False, add_generation_prompt=True, enable_thinking=False)
todo.sort(key=lambda r: len(r["user"]))          # gộp độ dài gần nhau cho đỡ đệm
with open(RAW, "a", encoding="utf-8") as f:
    for i in range(0, len(todo), BATCH):
        b = todo[i:i + BATCH]
        enc = tok([prompt(r) for r in b], return_tensors="pt", padding=True).to(model.device)
        with torch.no_grad():
            out = model.generate(**enc, max_new_tokens=MAX_NEW, do_sample=False)
        for r, o in zip(b, out):
            text = tok.decode(o[enc.input_ids.shape[1]:], skip_special_tokens=True)
            f.write(json.dumps({"item": r["item"], "output": text, "model": MODEL_ID, "revision": REV,
                                "inputs_sha256": SHA}, ensure_ascii=False) + "\n")
        f.flush()
        if (i // BATCH) % 10 == 0: print(i + len(b), "/", len(todo))
files.download(str(RAW))